# RARE ANIMALS IMAGE CLASSIFICATION - GROUP 12

#### **Group members**:
- **Yan Sidoryk** - 20222004
- **Henry Lewis** - 20222002
- **Abdul Rehman Khan** - 20231738
- **Lowie De Wever** - 20231733

## Importing libraries

In [ ]:
# Install all the packages
# !pip install -r requirements.txt

In [ ]:
# Standard library imports
import os
import math
from collections import Counter
import ast
import numpy as np
import pandas as pd
from tqdm import tqdm

# Visualization
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Image processing
from PIL import Image as PilImage

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# Deep learning 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential ###, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import L1, L2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam, Adagrad
from tensorflow.keras.applications import InceptionV3, EfficientNetB4

# Hyperparameter tuning
import keras_tuner as kt

# Pretrained models (and torch)
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel
import torch

In [ ]:
# Import custom functions
from custom_function import * 
%load_ext autoreload
%autoreload 2

In [ ]:
directory = 'rare_species/'

In [ ]:
# Cheak if GPU is detected
print(f"Cuda is availible: {torch.cuda.is_available()}")
print("GPUs detected:", tf.config.list_physical_devices('GPU'))

In [ ]:
metadata = pd.read_csv(f'{directory}metadata.csv')

## Data exploration

### Metadata exploration

#### Basic exploration

In [ ]:
metadata.info()

In [ ]:
metadata.head()

In [ ]:
metadata.describe(include='O')

In [ ]:
plot_phylum_counts(metadata)

In [ ]:
plot_family_counts(metadata)

In [ ]:
plot_taxonomic_hierarchy(metadata)

#### Retracting more information from the images

In [ ]:
# Extract more information about the images and save to the metadata
metadata['image_size'], metadata['color_channel'], metadata['format'], metadata['aspect_ratio'], \
metadata['min_val'], metadata['max_val'] = explore_image_files(directory + metadata['file_path'], explore_values=True)

# # Don't explore values to save time (1min)
# metadata['image_size'], metadata['color_channel'], metadata['format'], metadata['aspect_ratio'] = explore_image_files(directory + metadata['file_path'])

In [ ]:
# Compute width, height and pixel count
metadata['width'], metadata['height'] = zip(*metadata['image_size'])
metadata["pixel_count"] = metadata["width"] * metadata["height"]

In [ ]:
metadata.head()

In [ ]:
metadata.describe()

In [ ]:
smallest_largest_image(metadata, directory)

In [ ]:
# Based on previous discoveries set up potencial boudaries for image scaling
common_ratio = 4/3
width_max = 500       

In [ ]:
plot_image_size_scatter(metadata, common_ratio, width_max)

In [ ]:
# Calculate the final image size
image_size = (width_max, round(width_max/common_ratio))
print(f"Best image size to scale to is {image_size}")

In [ ]:
plot_color_chanels_counts(metadata)

### Images exploration

#### All images

In [ ]:
# Running this cell produces too many images and makes the notebook 6 times the size
# So we will avoid running it in the final submission, but it was used for data exploration
# plot_images_from_directory(directory, img_per_class=5) 

From plotting sample of the images we can see that have some unusial examples like:

- X-ray imagaes
- Images of signs
- Paintings
- Text extracts with no images
- Images unrelated to the class
- Maps
- Varying zoom / color / rotations

#### Grayscale and CMYK
Lets take a look at what are the different color formats

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)

# Get only greyscale images
temp_generator = datagen.flow_from_dataframe(
    dataframe=metadata[metadata.color_channel == 'L'],
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # Not shuffling
)

In [ ]:
plot_images_from_generator(temp_generator, num_batches=3)

We can see that most of Grayscale images are sketches, drawings, X-rays or just old photos. Some images retain at least the shape of the actual animal, but most of them are really bad.

In [ ]:
# Do the same for CMYK
temp_generator = datagen.flow_from_dataframe(
    dataframe=metadata[metadata.color_channel == 'CMYK'],
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # Not shuffling
)

In [ ]:
plot_images_from_generator(temp_generator, num_batches=3)

Most of the images apper to be x-rays of sculls of different types of monkeys and apes, which don't really help identify the animals by the picture, and will more likely only confuse the model. \
(and there are 2 different images for some reason: 1 with 2 parrots and another with a turtle)

In [ ]:
# See what is the class distribution amoung the greyscale images
unique, counts = np.unique(temp_generator.classes, return_counts=True)
counts_per_class = dict(zip(unique, counts))

idx_to_class = {v: k for k, v in temp_generator.class_indices.items()}

for idx, count in counts_per_class.items():
    print(f"{idx_to_class[idx]}: {count} images")

In [ ]:
# Changing good CMYK images to RGB in the metadata to keep them in the dataset
metadata.loc[metadata['family'].isin(['chordata_emydidae', 'chordata_psittacidae']) & 
             (metadata['color_channel'] == 'CMYK'), 'color_channel'] = 'RGB'

## Preprocessing

### Remove CMYK and Grayscale

In [ ]:
# Remove CMYK and Grayscale images from metadata (keeping the 2 good CMYK images)
print(f"Original metadata size: {len(metadata)}")
metadata = metadata[~metadata['color_channel'].isin(['CMYK', 'L'])].reset_index(drop=True)
print(f"After removing CMYK/grayscale: {len(metadata)}")

### Outlier detection

#### YOLO

In [ ]:
# Set up the parameters for the model
MODEL_ID = 'yolov8s.pt'
CONFIDENCE_THRESHOLD = 0.85
PERSON_CLASS_ID = 0

In [ ]:
# Run YOLO person detection
metadata['has_person'], metadata['person_confidence'], metadata['person_bbox'] = find_people_yolo(metadata, directory, MODEL_ID, CONFIDENCE_THRESHOLD, PERSON_CLASS_ID)

# Print the results
print(f"\n--- YOLO Results ---")
print(f"Images with people: {metadata['has_person'].sum()}")
print(f"Images without people: {(~metadata['has_person']).sum()}")

#### Imagenet detection

In [ ]:
# Load the model
imagenet_model = InceptionV3(weights='imagenet')

In [ ]:
# Run animal detection with pretrained model
metadata['has_animal'], metadata['animal_confidence'], metadata['predicted_class'], metadata['prediction_confidence'] = check_animal_presence(metadata, directory, imagenet_model)

# Print the results
print(f"\n--- ImageNet Results : ---")
print(f"Images with animals: {metadata['has_animal'].sum()}")
print(f"Images without animals: {(~metadata['has_animal']).sum()}")

#### Clip

In [ ]:
# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Define semantic categories
text_prompts = [
    # Good categories (KEEP) - indices 0-2
    "a photograph of an animal",
    "a wildlife photograph",
    "a photo of an animal in nature",
    
    # Bad categories (REMOVE) - indices 3+
    "an x-ray image",
    "a medical scan",
    "a drawing or sketch",
    "a map or diagram",
    "text document or book page",
    "a logo or icon",
    "a cooked meal",
    "prepared food",
    "a human",
    "a photo of a person",
    "a portrait of a human face",
    "people in a photograph",
    "a group of people",
    "a scientist or researcher",
]

# Compute text embeddings
text_inputs = clip_processor(text=text_prompts, return_tensors="pt", padding=True)
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

In [ ]:
# Run CLIP scoring on metadata
clip_results = compute_clip_scores_batch(metadata, directory, clip_processor, clip_model, text_prompts, text_features, batch_size=32)

# Add CLIP columns to metadata
metadata['clip_photo_score'] = clip_results['clip_photo_score']
metadata['clip_noise_score'] = clip_results['clip_noise_score']
metadata['clip_semantic_quality'] = clip_results['clip_semantic_quality']
metadata['clip_best_match'] = clip_results['clip_best_match']

#### Visualization / Comparisons

In [ ]:
plot_yolo_detections(metadata, directory, 100)

In [ ]:
plot_non_animal_images(metadata, directory, n_cols=10)

In [ ]:
plot_clip_outliers(metadata, directory, 100)

Short visualization summary:
- YOLOv8 person detection incorrectly flagged too many images of researchers holding animals, which we would like to keep.
- InceptionV3 flagged almost 40% of the images as "not containing animals", with a lot of misclassifications.
- CLIP gave the best results, identifying 640 outliers such as X-rays, diagrams, and text.

### Outlier removal

In [ ]:
# Remove the outliers based on the CLIP results
clip_outliers = metadata[metadata['clip_semantic_quality'] < 0].copy()
metadata = metadata[metadata['clip_semantic_quality'] >= 0].reset_index(drop=True)

In [ ]:
# Check what class has the least images (save the family name for later)
low_images_class = metadata['family'].value_counts().idxmin()
print(f"Total images after removing inconsistencies: {len(metadata)} out of {len(metadata) + len(clip_outliers)} ({len(clip_outliers)} removed)")
print(f"The lowest amount of images per class is '{low_images_class}' with {metadata['family'].value_counts().iloc[-1]} images")

### Generate embeddings

In [ ]:
# Load pre-trained EfficientNet without the top classification layer
embedding_model = EfficientNetB4(weights='imagenet', include_top=False, pooling='avg')
print(f"Embedding dimension: {embedding_model.output_shape[-1]}")

In [ ]:
# Add embedings to metadata
metadata['embedding'] = generate_embeddings(metadata, directory, image_size, embedding_model)

### Remove inconsistencies

In [ ]:
# Split the metadata not to leak the test embeddings
train_df, test_df = train_test_split(
  metadata,
  test_size=0.15,                 
  stratify=metadata['family'],    # Stratify by the target
  shuffle=True,
  random_state=42
)

In [ ]:
# Calculate distance to centroids (centroids are based only on train embeddings)
train_df['distance_to_centroid'], test_df['distance_to_centroid'] = distance_to_centroid(train_df, test_df)

In [ ]:
plot_furthest_images_batched(directory, train_df, 100)

In [ ]:
# Identify outliers based on distance to the centroid
threshold = train_df['distance_to_centroid'].quantile(0.95) 

# Copy outliers in case we need them later
outliers_train = train_df[train_df['distance_to_centroid'] > threshold].copy()
outliers_test = test_df[test_df['distance_to_centroid'] > threshold].copy()     

# Remove outliers from the main dataset
train_df = train_df[train_df['distance_to_centroid'] <= threshold].copy()
test_df = test_df[test_df['distance_to_centroid'] <= threshold].copy()

print(f"Removed {len(outliers_train)} outliers from train data")
print(f"Removed {len(outliers_test)} outliers from test data")

In [ ]:
# See what class has the least images (save the family name for later)
low_images_class = train_df['family'].value_counts().idxmin()
print(f"Total images after removing inconsistencies: {len(train_df) + len(test_df)} (out of {len(metadata)})")
print(f"Images left for training: {len(train_df)}. Images left for testing: {len(test_df)}.")
print(f"The lowest amount of images per class is '{low_images_class}' with {train_df['family'].value_counts().iloc[-1]} images for training and {test_df['family'].value_counts().iloc[-1]} images for testing")

In [ ]:
# Recalculate centroids
train_df.drop('distance_to_centroid', axis=1, inplace=True)
test_df.drop('distance_to_centroid', axis=1, inplace=True)
train_df['distance_to_centroid'], test_df['distance_to_centroid'] = distance_to_centroid(train_df, test_df)

In [ ]:
plot_furthest_images_batched(directory, train_df, 100)

In [ ]:
# Identify outliers based on distance to centroid (higher threshold this time)
threshold = train_df['distance_to_centroid'].quantile(0.975) 

# Copy outliers in case we need them later
outliers_train = pd.concat([outliers_train, train_df[train_df['distance_to_centroid'] > threshold]], ignore_index=True)
outliers_test = pd.concat([outliers_test, test_df[test_df['distance_to_centroid'] > threshold]], ignore_index=True)   

# Remove outliers from the main dataset
train_df = train_df[train_df['distance_to_centroid'] <= threshold].copy()
test_df = test_df[test_df['distance_to_centroid'] <= threshold].copy()

print(f"Removed Total of {len(outliers_train)} outliers from train data")
print(f"Removed Total of {len(outliers_test)} outliers from test data")

In [ ]:
# See what class has the least images (save the family name for later)
low_images_class = train_df['family'].value_counts().idxmin()
print(f"Total images after removing inconsistencies: {len(train_df) + len(test_df)} (out of {len(metadata)})")
print(f"Images left for training: {len(train_df)}. Images left for testing: {len(test_df)}.")
print(f"The lowest amount of images per class is '{low_images_class}' with {train_df['family'].value_counts().iloc[-1]} images for training and {test_df['family'].value_counts().iloc[-1]} images for testing")

In [ ]:
plot_furthest_images_batched(directory, train_df, 30)

In [ ]:
plot_furthest_images_batched(directory, train_df[train_df.family == low_images_class], 20)

### Split the data

In [ ]:
# Set a part of train for validation
train_df, val_df = train_test_split(
  train_df,
  test_size=0.1,
  stratify=train_df['family'],
  shuffle=True,
  random_state=42
)

# Print the final proportions of the split
total_data_len = len(train_df) + len(val_df) + len(test_df)     # Amout of data after cleaning
print(f"Training  : {len(train_df)} ({round(len(train_df)/total_data_len*100)}%) \
        \nValidation: {len(val_df)} ({round(len(val_df)/total_data_len*100)}%)  \
        \nTesting   : {len(test_df)} ({round(len(test_df)/total_data_len*100)}%) \
        \nTotal     : {total_data_len}")

In [ ]:
# Set up train generator with data augmentatin
train_datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    fill_mode='nearest',
    # width_shift_range=0.10,
    # height_shift_range=0.10,
)

# Validation/Test generator (no augmentation)
test_datagen = ImageDataGenerator()

In [ ]:
# Create train from train_datagen
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,     # Transform to image_size
    batch_size=32,
    class_mode='sparse',        # Use sparse for integer labels
    color_mode='rgb',           # Convert all the images to RGB
    shuffle=True,               # Shuffle the train set
    seed=42
)

# And validation and test from test_datagen
val_generator = test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',  
    color_mode='rgb',
    shuffle=False, 
    seed=42
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',  
    color_mode='rgb',
    shuffle=False,              # Not shuffling for reproducibility
    seed=42
    )

In [ ]:
# Check one batch for any inconsistencies
images, labels = next(train_generator)

print(f"VALIDATION CHECKS:")
print(f"Image batch shape: {images.shape}")
print(f"Image value range: [{images.min():.3f}, {images.max():.3f}]")
print(f"Labels shape: {labels.shape}")
print(f"Labels range: [{labels.min()}, {labels.max()}]")
print(f"Unique labels in batch: {len(np.unique(labels))}")
print(f"Any all-zero images: {(images.sum(axis=(1,2,3)) == 0).any()}")

In [ ]:
plot_datagen_sample(train_generator)

## Modeling


In [ ]:
N_CLASSES = len(train_generator.class_indices)
print(f"Number of classes: {N_CLASSES}")

### Hyperband parameter search

In [ ]:
def build_model(hp):

    # Get the base model set up
    base_model = EfficientNetB4(
        weights='imagenet',
        include_top=False,
        input_shape=(*image_size, 3) 
    )
    # Freeze the base model
    base_model.trainable = False

    with tf.device('/GPU:0'):   # Move the model to gpu
        model = Sequential([
            base_model,
            GlobalAveragePooling2D(),
            BatchNormalization()
        ])

    # Tune the number of hidden layers
    for i in range(1, hp.Int("num_layers", 2, 4)):  # 1, 2 or 3 layers (excluding the final classification layer)
        model.add(
            keras.layers.Dense(
                units=hp.Int("units_" + str(i), min_value=400, max_value=1000, step=200),
                activation="relu")
            )

        # Tune the Dropout
        model.add(keras.layers.Dropout(hp.Float("dropout_" + str(i), 0, 0.5, step=0.1)))

    # Add classification layer
    model.add(keras.layers.Dense(units=N_CLASSES, activation="softmax"))

    # Add optimizer choice
    optimizer_name = hp.Choice("optimizer", ["adam", "adagrad"])

    if optimizer_name == "adam":
        optimizer = Adam(learning_rate=0.001)
    else:
        optimizer = Adagrad(learning_rate=0.01)

    # Compile the model
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=["accuracy"]
    )
    
    return model

In [ ]:
# Create the tuner
tuner = kt.Hyperband(
    build_model,
    objective='val_loss',
    max_epochs=10,
    factor=3,
    directory='keras_tuner',
    project_name='b4_test3'
)

In [ ]:
tuner.search_space_summary()

In [ ]:
# Set up callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

# Calculate the class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

In [ ]:
# Run the tuner
tuner.search(train_generator, epochs=10, batch_size=400, class_weight=class_weight_dict, validation_data=val_generator, callbacks=[callbacks])

In [ ]:
# Check the best hyperparameters
best_hp=tuner.get_best_hyperparameters()[0]
print(best_hp.values)

So the final **best model** (to add on top of pretrained bone) is:

- One Dense layer with **600 neurons**

- Dropout of **0.4**

- And the optimizer is  **adagrad**

### Train the best model

In [ ]:
# # Generate model with best parameters
# tuned_B4 = build_model(best_hp)

In [ ]:
# Get the base model set up
base_model_B4 = EfficientNetB4(
    weights='imagenet',
    include_top=False,
    input_shape=(*image_size, 3) 
)

# Freeze the base model
base_model_B4.trainable = False

In [ ]:
# Calculate the class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

In [ ]:
# Build the classifier
with tf.device('/GPU:0'):   # Move the model to gpu
    tuned_model = Sequential([
        base_model_B4,
        GlobalAveragePooling2D(),
        BatchNormalization(),

        Dense(600),
        Dropout(0.4),
        
        Dense(N_CLASSES, activation='softmax')
    ])

In [ ]:
# See the parameter counts
print("Total params:", tuned_model.count_params())
print("Non-trainable params:", sum([tf.size(w).numpy() for w in tuned_model.non_trainable_weights]))
print("Trainable params:", sum([tf.size(w).numpy() for w in tuned_model.trainable_weights]))

In [ ]:
# Compile the model
tuned_model.compile(
    optimizer=Adagrad(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',     
    metrics=['accuracy'] 
)

# Set up callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.1,
        patience=2,
        min_lr=1e-5,
        verbose=1
    )
]

# Fit the model
history_tuned = tuned_model.fit(
    train_generator,
    epochs=20,
    batch_size=400,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
plot_model_results(history_tuned, 'loss')

In [ ]:
plot_model_results(history_tuned, 'accuracy')

### Fine tuning

In [ ]:
# Clone the model
fine_tuned_model = tf.keras.models.clone_model(tuned_model)
fine_tuned_model.set_weights(tuned_model.get_weights())

In [ ]:
# Unfreeze last 5 layers of base model
base_model_finetune = fine_tuned_model.get_layer('efficientnetb4')

for layer in base_model_finetune.layers[-5:]:
    print(layer)
    # Keep BatchNormalization frozen
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

In [ ]:
# See the parameter counts
print("Total params:", fine_tuned_model.count_params())
print("Non-trainable params:", sum([tf.size(w).numpy() for w in fine_tuned_model.non_trainable_weights]))
print("Trainable params:", sum([tf.size(w).numpy() for w in fine_tuned_model.trainable_weights]))

In [ ]:
# Compile the model
fine_tuned_model.compile(
    optimizer=Adam(learning_rate=1e-7),
    loss='sparse_categorical_crossentropy',     
    metrics=['accuracy'] 
)

# Set up callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=2,
        restore_best_weights=True,
        verbose=1
    )
]

# Fit the model
history_fine_tuned = fine_tuned_model.fit(
    train_generator,
    epochs=10,
    batch_size=400,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
plot_model_results(history_tuned, 'loss', history_fine_tuned)

In [ ]:
plot_model_results(history_tuned, 'accuracy', history_fine_tuned)

## Results

In [ ]:
test_results = fine_tuned_model.evaluate(test_generator, verbose=1)
print(f"Test Accuracy: {test_results[1]:.4f}")
print(f"Test Loss: {test_results[0]:.4f}")

In [ ]:
# Get predictions
y_pred = fine_tuned_model.predict(test_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)

# Get true labels
y_true = test_generator.classes

In [ ]:
# Calculate metrics
acc = accuracy_score(y_true, y_pred_classes)
f1_micro = f1_score(y_true, y_pred_classes, average='micro')
f1_macro = f1_score(y_true, y_pred_classes, average='macro')
f1_weighted = f1_score(y_true, y_pred_classes, average='weighted')

# Print the metrics
print("--- Model performance ---")
print(f"Accuracy           : {acc:.4f}")
print(f"F1 Score (Micro)   : {f1_micro:.4f}")
print(f"F1 Score (Macro)   : {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")

In [ ]:
# Check the confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)
plot_confusion_matrix(cm)

In [ ]:
# Get the class names for plotting
class_names = sorted(list(metadata['family'].unique()))

In [ ]:
plot_misclassifications(test_generator, y_pred_classes, y_true, class_names, train_df, directory, 10)

## Bonus: Using Metadata to help the model

In [ ]:
# Create mapping from family to sparse ID
family_to_id = train_generator.class_indices

# Create mapping from phylum to family
phylum_to_families = {}

for phylum, group in train_df.groupby("phylum"):
    families = group["family"].unique()                
    family_ids = [family_to_id[f] for f in families]   # Convert to sparse IDs
    phylum_to_families[phylum] = family_ids

In [ ]:
# Get predictions with metadata
predictions_with_metadata = predict_with_metadata(tuned_model, test_generator, test_df, phylum_to_families)

# Get true labels
y_true = test_generator.classes

In [ ]:
# Calculate metrics
acc = accuracy_score(y_true, predictions_with_metadata)
f1_micro = f1_score(y_true, predictions_with_metadata, average='micro')
f1_macro = f1_score(y_true, predictions_with_metadata, average='macro')
f1_weighted = f1_score(y_true, predictions_with_metadata, average='weighted')


# Print the metrics
print("--- Model performance WITH metadata ---")
print(f"Accuracy           : {acc:.4f}")
print(f"F1 Score (Micro)   : {f1_micro:.4f}")
print(f"F1 Score (Macro)   : {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")